# Prévision des hospitalisations COVID-19

Nous travaillons sur le taux hebdomadaire d'hospitalisation COVID-19 aux États-Unis (réseau COVID-NET), exprimé pour 100 000 habitants, du 7 mars 2020 au 10 octobre 2026. Le fichier utilisé est celui obtenu après le nettoyage du jeu de données.

Nous suivons la démarche de Box-Jenkins : on observe la série, on étudie sa stationnarité, on choisit les ordres du modèle à partir de l'ACF et de la PACF, puis on compare les prévisions de plusieurs modèles sur une période de test. Chaque modèle est comparé à deux baselines simples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

## 1. La série

On fixe une fréquence hebdomadaire (chaque samedi) avec `asfreq`. Le modèle raisonne en pas de temps : le lag 1 correspond à une semaine et le lag 52 à une année. Si une semaine manquait, elle apparaîtrait comme valeur manquante au lieu d'être ignorée.

In [ ]:
df = pd.read_csv("hospitalisations_hebdo_clean.csv", parse_dates=["date"], index_col="date")
df = df[["covid"]]
y = df["covid"].asfreq("W-SAT")

print("Période :", y.index.min().date(), "→", y.index.max().date())
print("Nombre de semaines :", len(y))
print("Valeurs manquantes :", y.isna().sum())

y.plot(figsize=(12, 4), title="Taux d'hospitalisation hebdomadaire COVID-19 (pour 100 000 hab.)")
plt.grid(alpha=0.3)
plt.show()

La série contient 345 semaines sans valeur manquante.

On observe des pics qui reviennent chaque hiver, ce qui indique une saisonnalité annuelle. Ces pics sont de plus en plus bas au fil des années, ce qui correspond à une tendance à la baisse. La période 2020-2022 est différente du reste : les vagues sont beaucoup plus fortes et irrégulières, avec un maximum d'environ 36 en janvier 2022 (vague Omicron). La baisse des pics est probablement liée à l'immunité acquise, mais nos données ne permettent pas de le vérifier.

## 2. Stationnarité

Un modèle ARIMA suppose que la série est stationnaire, c'est-à-dire que sa moyenne et sa variance restent stables dans le temps. On commence par regarder l'ACF et la PACF de la série brute.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y, lags=60, ax=axes[0], title="ACF de y")
plot_pacf(y, lags=60, ax=axes[1], title="PACF de y", method="ywm")
for ax in axes:
    ax.axvline(52, color="red", ls="--", lw=1)
plt.tight_layout()
plt.show()

L'ACF décroît lentement : une valeur élevée reste élevée pendant plusieurs semaines. C'est le signe d'une forte persistance et d'une série probablement non stationnaire. L'ACF remonte autour du lag 52, ce qui confirme la saisonnalité annuelle.

On ne choisit pas encore les ordres p et q à partir de ces graphiques, car sur une série non stationnaire l'ACF est dominée par la persistance.

Pour confirmer avec un test, on utilise ADF et KPSS. Ces deux tests ont des hypothèses opposées. Pour ADF, l'hypothèse nulle est que la série n'est pas stationnaire, donc une p-value inférieure à 0.05 indique une série stationnaire. Pour KPSS, c'est l'inverse : l'hypothèse nulle est que la série est stationnaire, donc une p-value inférieure à 0.05 indique une série non stationnaire.

In [ ]:
def tests_stationnarite(serie, nom):
    serie = serie.dropna()
    p_adf = adfuller(serie, autolag="AIC")[1]
    p_kpss = kpss(serie, regression="c", nlags="auto")[1]
    print(f"{nom:<4} ADF p-value = {p_adf:.4f}   KPSS p-value = {p_kpss:.4f}")

tests_stationnarite(y, "y")
tests_stationnarite(y.diff(), "Δy")

Sur la série brute, les deux tests se contredisent. ADF donne une p-value de 0.003 et conclut que la série est stationnaire, alors que KPSS donne une p-value inférieure à 0.01 et conclut le contraire. ADF voit une série qui revient vers un niveau bas après chaque pic, tandis que KPSS détecte les changements de niveau entre la période 2020-2022 et la suite.

Après une différenciation, les deux tests sont d'accord : la série différenciée est stationnaire. On retient donc d = 1.

## 3. Choix des ordres p et q

La série différenciée correspond à la variation d'une semaine à l'autre. On la calcule ici uniquement pour lire l'ACF et la PACF, car c'est cette série que le modèle ARIMA va réellement modéliser. Pour ajuster le modèle, on donnera directement y avec d = 1, et le modèle se chargera de la différenciation.

Pour rappel, une PACF qui coupe après le lag p avec une ACF qui décroît suggère un AR(p). Une ACF qui coupe après le lag q avec une PACF qui décroît suggère un MA(q).

In [ ]:
dy = y.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(dy, lags=60, ax=axes[0], title="ACF de Δy")
plot_pacf(dy, lags=60, ax=axes[1], title="PACF de Δy", method="ywm")
for ax in axes:
    ax.axvline(52, color="red", ls="--", lw=1)
plt.tight_layout()
plt.show()

L'ACF vaut environ 0.70 au lag 1. Quand les hospitalisations augmentent une semaine, elles ont tendance à continuer d'augmenter la semaine suivante. La PACF est d'environ -0.40 au lag 2 puis s'arrête : cette hausse finit par ralentir, ce qui correspond au moment où une vague approche de son pic.

L'ACF devient négative autour des lags 8 à 12. Une montée est souvent suivie d'une descente une dizaine de semaines plus tard, ce qui correspond à peu près à la durée d'une demi-vague. On voit aussi un pic d'environ 0.24 au lag 52 : la saisonnalité est toujours présente après la différenciation, ce qui justifiera un modèle saisonnier.

La PACF coupe après le lag 2, ce qui suggère un ARIMA(2,1,0). L'ACF coupe aussi assez vite, donc un ARIMA(0,1,2) est également possible. On départage ces candidats avec l'AIC.

## 4. Découpage entraînement et test

Le découpage se fait avant tout ajustement et respecte l'ordre chronologique. Le modèle est entraîné de mars 2020 à fin septembre 2025 (291 semaines) et évalué sur la dernière saison complète, d'octobre 2025 à octobre 2026 (54 semaines).

Les modèles prévoient les 54 semaines du test en une seule fois, à partir de la fin de la période d'entraînement. On mesure l'erreur avec la RMSE, qui pénalise fortement les grosses erreurs, et la MAE, qui donne l'erreur moyenne en points de taux.

In [ ]:
TEST_START = pd.Timestamp("2025-10-04")

y_train = y[y.index < TEST_START]
y_test = y[y.index >= TEST_START]
H = len(y_test)

print(f"Entraînement : {len(y_train)} semaines")
print(f"Test : {H} semaines")

In [ ]:
def rmse(vrai, prevu):
    return np.sqrt(np.mean((np.asarray(vrai) - np.asarray(prevu)) ** 2))

def mae(vrai, prevu):
    return np.mean(np.abs(np.asarray(vrai) - np.asarray(prevu)))

previsions = {}

def comparer(nom, prev):
    previsions[nom] = prev

    plt.figure(figsize=(12, 4))
    plt.plot(y[-80:], color="black", lw=2, label="Observé")
    plt.plot(previsions["Naïf"], color="grey", ls="--", label="Naïf")
    plt.plot(previsions["Naïf saisonnier"], color="tab:orange", ls=":", lw=2, label="Naïf saisonnier")
    plt.plot(prev, color="tab:blue", lw=2, label=nom)
    plt.axvline(TEST_START, color="grey", lw=0.8)
    plt.title(nom)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    for n in ["Naïf", "Naïf saisonnier", nom]:
        print(f"{n:<40} RMSE = {rmse(y_test, previsions[n]):.2f}   MAE = {mae(y_test, previsions[n]):.2f}")

## 5. Baselines

Avant de tester des modèles plus élaborés, on définit deux prévisions de référence. La prévision naïve répète la dernière valeur connue pendant toute la période de test. La prévision naïve saisonnière reprend, pour chaque semaine, la valeur de la même semaine un an plus tôt. Un modèle qui ne fait pas mieux que ces deux références n'a pas d'intérêt.

In [ ]:
previsions["Naïf"] = pd.Series(y_train.iloc[-1], index=y_test.index)

derniere_annee = y_train.iloc[-52:].values
previsions["Naïf saisonnier"] = pd.Series(
    [derniere_annee[h % 52] for h in range(H)], index=y_test.index
)

plt.figure(figsize=(12, 4))
plt.plot(y[-80:], color="black", lw=2, label="Observé")
plt.plot(previsions["Naïf"], color="grey", ls="--", label="Naïf")
plt.plot(previsions["Naïf saisonnier"], color="tab:orange", ls=":", lw=2, label="Naïf saisonnier")
plt.axvline(TEST_START, color="grey", lw=0.8)
plt.title("Baselines")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for n in ["Naïf", "Naïf saisonnier"]:
    print(f"{n:<20} RMSE = {rmse(y_test, previsions[n]):.2f}   MAE = {mae(y_test, previsions[n]):.2f}")

Le naïf obtient une RMSE de 1.15 et le naïf saisonnier une RMSE de 1.23. Le naïf reste bloqué à 1.9, la dernière valeur de l'entraînement, ce qui est trop haut pour presque toute la saison. Le naïf saisonnier a la bonne forme, avec un pic en janvier et un creux en été, mais il recopie un pic de 4.3 alors que celui de cette année ne dépasse pas 2.1. Comme les pics diminuent chaque année, il surestime tout.

## 6. ARIMA

On compare plusieurs ordres autour des candidats identifiés avec l'ACF et la PACF. L'AIC mesure le compromis entre la qualité d'ajustement et le nombre de paramètres : plus il est faible, meilleur est le modèle. Il ne se compare qu'entre modèles qui ont le même ordre de différenciation.

La fonction `SARIMAX` de statsmodels permet d'ajuster un simple ARIMA lorsqu'on ne précise ni partie saisonnière ni variables externes.

In [ ]:
for ordre in [(1, 1, 0), (2, 1, 0), (3, 1, 0), (2, 1, 1), (0, 1, 2)]:
    res = SARIMAX(y_train, order=ordre).fit(disp=False)
    print(f"ARIMA{ordre}   AIC = {res.aic:.1f}")

L'ARIMA(1,1,0) est nettement moins bon que les autres (AIC de 822 contre environ 775), ce qui montre qu'un seul retard ne suffit pas. Les modèles à deux ou trois paramètres obtiennent des AIC très proches. L'ARIMA(0,1,2) a le meilleur score (771.3), devant l'ARIMA(2,1,0) (775.0). Comme ajouter un troisième paramètre n'apporte presque rien, on garde les deux modèles à deux paramètres.

In [ ]:
res_arima = SARIMAX(y_train, order=(2, 1, 0)).fit(disp=False)
print(res_arima.summary())

Les deux coefficients de l'ARIMA(2,1,0) sont significatifs (p < 0.001). Le premier vaut 0.97 et le second -0.40, ce qui correspond aux signes observés sur la PACF. La variation d'une semaine prolonge celle de la semaine précédente, puis elle est freinée par celle d'il y a deux semaines. Ce mécanisme produit une dynamique de vague.

Le test de Ljung-Box (p = 0.48) ne montre pas d'autocorrélation restante dans les résidus. En revanche, les résidus ne sont pas normaux (Jarque-Bera, p < 0.001) et leur variance n'est pas constante (H = 0.06). Les erreurs sont beaucoup plus grandes pendant les vagues de 2020-2022 que sur la période récente. Les prévisions restent utilisables, mais les intervalles de confiance seront peu fiables.

In [ ]:
res = SARIMAX(y_train, order=(2, 1, 0)).fit(disp=False)
comparer("ARIMA(2,1,0)", res.get_forecast(steps=H).predicted_mean)

In [ ]:
res = SARIMAX(y_train, order=(0, 1, 2)).fit(disp=False)
comparer("ARIMA(0,1,2)", res.get_forecast(steps=H).predicted_mean)

Les deux ARIMA font mieux que les baselines : la RMSE est de 0.76 pour l'ARIMA(2,1,0) et de 0.65 pour l'ARIMA(0,1,2). Cependant, leurs prévisions sont presque plates. C'est un comportement normal pour un ARIMA prévu à long terme : le modèle ne connaît pas les chocs futurs et les remplace par zéro. Avec d = 1, la variation prévue tend vers zéro et le niveau reste constant. Le modèle ne contient aucune information sur la saison et ne peut donc pas prévoir le pic de janvier.

Les ARIMA gagnent surtout parce que leur niveau constant, autour de 1.3, est plus proche du niveau moyen du test que celui du naïf. Il faut donc un modèle qui garde ce niveau tout en ajoutant la saisonnalité.

## 7. SARIMA

Un SARIMA ajoute une partie saisonnière à l'ARIMA. Il s'écrit SARIMA(p,d,q)(P,D,Q)s. La première parenthèse décrit la dynamique d'une semaine à l'autre, comme dans l'ARIMA. La seconde fait la même chose d'une année à l'autre : P relie la semaine à la même semaine de l'année précédente, D applique une différence saisonnière (la valeur moins celle d'il y a 52 semaines) et Q corrige avec l'erreur commise un an plus tôt. Ici s = 52, ce qui correspond au pic observé au lag 52.

On garde la partie non saisonnière du meilleur ARIMA, (0,1,2), et on teste plusieurs parties saisonnières. L'AIC n'est pas comparable entre D = 0 et D = 1, car le modèle avec D = 1 travaille sur une série différente.

In [ ]:
modeles_sarima = {}

for saison in [(1, 0, 0), (0, 0, 1), (1, 0, 1), (0, 1, 1)]:
    res = SARIMAX(y_train, order=(0, 1, 2), seasonal_order=saison + (52,)).fit(disp=False)
    modeles_sarima[saison] = res
    print(f"SARIMA(0,1,2){saison}52   AIC = {res.aic:.1f}")

Parmi les modèles avec D = 0, les trois candidats ont des AIC presque identiques (entre 761.5 et 762.4). On choisit le plus simple, SARIMA(0,1,2)(1,0,0)52, qui n'a qu'un seul paramètre saisonnier. Il gagne environ 10 points d'AIC par rapport à l'ARIMA(0,1,2), ce qui montre que la saisonnalité améliore réellement le modèle.

Le modèle avec D = 1 a un AIC beaucoup plus bas (681.3), mais ce chiffre n'est pas comparable aux autres. Seules les prévisions permettent de le juger.

In [ ]:
comparer("SARIMA(0,1,2)(1,0,0)52", modeles_sarima[(1, 0, 0)].get_forecast(steps=H).predicted_mean)

Le SARIMA(0,1,2)(1,0,0)52 obtient une RMSE de 0.38, soit environ 40 % d'erreur en moins que le meilleur ARIMA et deux tiers de moins que le naïf. Sa prévision reste proche du niveau observé et remonte légèrement à la fin de la période. Il ne reproduit pas vraiment le pic de janvier, mais il évite de le surestimer.

In [ ]:
comparer("SARIMA(0,1,2)(0,1,1)52", modeles_sarima[(0, 1, 1)].get_forecast(steps=H).predicted_mean)

Le modèle avec D = 1 est le seul à avoir la bonne forme : une montée à l'automne, un pic en janvier et une baisse au printemps. Mais il prévoit un pic proche de 12 alors que la valeur observée est d'environ 2, et il descend même sous zéro au printemps. Sa RMSE de 3.09 est la plus mauvaise de tous les modèles.

Avec la différence saisonnière, le modèle reproduit les variations des hivers précédents, y compris les très fortes hausses de 2021 et 2022. Comme le modèle est additif, il ajoute ces variations en points au niveau actuel, beaucoup plus bas, et rien ne l'empêche de produire des valeurs négatives. On voit aussi que le meilleur AIC ne garantit pas la meilleure prévision.

## 8. SARIMAX avec des termes de Fourier

Au lieu de faire référence à la valeur d'il y a 52 semaines, on peut décrire directement la forme de la saison avec des sinus et des cosinus de période 365.25 / 7, soit environ 52.18 semaines. Cette période tient compte des années de 53 semaines. Avec K = 1, la saison est une vague simple. Plus K augmente, plus le pic peut être étroit.

Ces sinus et cosinus sont ajoutés comme variables externes. On peut les calculer pour n'importe quelle date future, ce qui est nécessaire pour faire des prévisions. La partie ARIMA(0,1,2) s'occupe du reste de la dynamique.

In [ ]:
PERIODE = 365.25 / 7
t = np.arange(len(y))

def fourier(K):
    X = pd.DataFrame(index=y.index)
    for k in range(1, K + 1):
        X[f"sin{k}"] = np.sin(2 * np.pi * k * t / PERIODE)
        X[f"cos{k}"] = np.cos(2 * np.pi * k * t / PERIODE)
    return X

for K in [1, 2, 3, 4]:
    X = fourier(K)
    res = SARIMAX(y_train, order=(0, 1, 2), exog=X.loc[y_train.index]).fit(disp=False)
    print(f"Fourier K={K}   AIC = {res.aic:.1f}")

In [ ]:
K = 3
X = fourier(K)

res_fourier = SARIMAX(y_train, order=(0, 1, 2), exog=X.loc[y_train.index]).fit(disp=False)
print(res_fourier.summary())

comparer("SARIMAX(0,1,2) + Fourier K=3", res_fourier.get_forecast(steps=H, exog=X.loc[y_test.index]).predicted_mean)

Le meilleur AIC est obtenu avec K = 3 (761.4), mais la RMSE sur le test est de 2.92. Cinq coefficients de Fourier sur six ne sont pas significatifs, et leurs intervalles de confiance sont très larges. Le modèle doit trouver une seule forme de saison pour des hivers très différents, avec des pics proches de 35 en 2022 et de 2 à 4 les années suivantes. La saison estimée a donc une amplitude trop forte pour l'hiver 2025-2026.

Ce modèle et le SARIMA(0,1,2)(1,0,0)52 ont pratiquement le même AIC, mais leurs RMSE sur le test sont très différentes (0.38 contre 2.92). Un bon ajustement sur les données d'entraînement ne garantit pas une bonne prévision. Le SARIMA s'en sort mieux parce que sa partie saisonnière s'appuie surtout sur l'année précédente, plus calme, alors que Fourier impose la même forme à toutes les années.

## 9. Holt-Winters

Les résultats précédents montrent un problème commun : la saisonnalité est apprise en points, alors que les pics diminuent d'année en année. Une saisonnalité multiplicative, exprimée en pourcentage du niveau, devrait mieux convenir.

Holt-Winters est un modèle de lissage exponentiel. Il calcule une moyenne pondérée du passé en donnant plus de poids aux semaines récentes. Il comporte un niveau, une tendance (ici amortie pour éviter de prolonger une pente indéfiniment) et une saison qui est réajustée à chaque semaine. On teste une saison additive et une saison multiplicative. La version multiplicative est possible car la série est toujours positive.

In [ ]:
hw_add = ExponentialSmoothing(
    y_train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=52
).fit()

comparer("Holt-Winters additif", hw_add.forecast(H))

Résultats à compléter.

In [ ]:
hw_mul = ExponentialSmoothing(
    y_train, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=52
).fit()

comparer("Holt-Winters multiplicatif", hw_mul.forecast(H))

Résultats à compléter.

## 10. Comparaison de tous les modèles

In [ ]:
resultats = pd.DataFrame({
    nom: {"RMSE": rmse(y_test, prev), "MAE": mae(y_test, prev)}
    for nom, prev in previsions.items()
}).T.sort_values("RMSE").round(2)

resultats

## 11. Conclusion

La saisonnalité est indispensable pour cette série. L'ARIMA seul produit une prévision presque plate, alors que le SARIMA(0,1,2)(1,0,0)52 réduit nettement l'erreur (RMSE de 0.38 contre 1.15 pour le naïf). Ce modèle reste toutefois prudent et ne reproduit pas vraiment le pic de janvier.

Plusieurs approches échouent pour la même raison. Le naïf saisonnier, le SARIMA avec différence saisonnière et le modèle avec termes de Fourier surestiment l'hiver 2025-2026, car l'amplitude de la saison a fortement diminué depuis les vagues de 2020-2022. Ces résultats montrent aussi qu'un bon AIC sur l'entraînement ne garantit pas une bonne prévision.

Holt-Winters multiplicatif est pensé pour ce problème, puisque sa saison est exprimée en proportion du niveau. Son intérêt reste à confirmer avec les résultats.

Cette analyse a plusieurs limites. L'évaluation repose sur une seule saison de test, et l'hiver 2025-2026 a été plutôt doux, ce qui favorise les modèles prudents. Les résidus des modèles ont une variance qui n'est pas constante. Pour aller plus loin, on pourrait entraîner les modèles uniquement sur la période après 2022, ou évaluer des prévisions à une semaine, plus proches de l'usage réel en santé publique.